<a href="https://colab.research.google.com/github/sararahman1729/Custom-Loss-Function/blob/Tiny-ImageNet/tiny_imagenet.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("akash2sharma/tiny-imagenet")

print("Path to dataset files:", path)

100%|██████████| 474M/474M [00:24<00:00, 20.6MB/s]

Extracting files...


Path to dataset files: /root/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1


In [ ]:
!mv /root/.cache/kagglehub/datasets/akash2sharma/tiny-imagenet/versions/1 .

In [ ]:
!mv ./1/tiny-imagenet-200 .

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models
import numpy as np
import copy
from sklearn.metrics import confusion_matrix, precision_score, recall_score
import matplotlib.pyplot as plt
import os
import shutil

# Define custom loss with entropy regularization and self-distillation
class CrossEntropyWithSelfDistillation(nn.Module):
    def __init__(self, alpha=0.5, beta=0.1, temperature=3.0):
        super(CrossEntropyWithSelfDistillation, self).__init__()
        self.alpha = alpha
        self.beta = beta
        self.temperature = temperature

    def update_hyperparameters(self, alpha, beta, temperature):
        self.alpha = alpha
        self.beta = beta
        self.temperature = temperature

    def forward(self, student_outputs, teacher_outputs, targets):
        # Cross-entropy loss with true labels
        ce_loss = nn.CrossEntropyLoss()(student_outputs, targets)

        # Entropy regularization on student outputs
        entropy_loss = -torch.sum(torch.softmax(student_outputs, dim=1) * torch.log_softmax(student_outputs, dim=1), dim=1).mean()

        # Distillation loss with teacher's soft predictions
        distillation_loss = nn.KLDivLoss(reduction='batchmean')(
            torch.log_softmax(student_outputs / self.temperature, dim=1),
            torch.softmax(teacher_outputs / self.temperature, dim=1)
        ) * (self.temperature ** 2)

        # Total loss
        total_loss = (1 - self.alpha) * ce_loss + self.beta * entropy_loss + self.alpha * distillation_loss
        return total_loss

# Training function with self-distillation
def train_model_with_distillation(model, dataloaders, criterion, optimizer, num_epochs, device='cuda'):
    teacher_model = copy.deepcopy(model)  # Teacher model is a frozen copy
    teacher_model.eval()
    best_model_wts = copy.deepcopy(model.state_dict())
    best_acc = 0.0
    best_cm = None

    for epoch in range(num_epochs):
        # Dynamically update alpha, beta, and temperature
        criterion.update_hyperparameters(alpha=0.1 + 0.1 * epoch, beta=0.05 + 0.025 * epoch, temperature=3.0)

        print(f'Epoch [{epoch+1}/{num_epochs}], Alpha: {criterion.alpha:.4f}, Beta: {criterion.beta:.4f}, Temperature: {criterion.temperature:.1f}')

        for phase in ['train', 'val']:
            if phase == 'train':
                model.train()
            else:
                model.eval()

            running_loss = 0.0
            running_corrects = 0
            all_preds = []
            all_labels = []

            for inputs, labels in dataloaders[phase]:
                inputs, labels = inputs.to(device), labels.to(device)
                optimizer.zero_grad()

                with torch.set_grad_enabled(phase == 'train'):
                    # Student model's forward pass
                    student_outputs = model(inputs)
                    _, preds = torch.max(student_outputs, 1)

                    # Teacher model's outputs (frozen)
                    with torch.no_grad():
                        teacher_outputs = teacher_model(inputs)

                    # Compute loss using both teacher and student outputs
                    loss = criterion(student_outputs, teacher_outputs, labels)

                    if phase == 'train':
                        loss.backward()
                        optimizer.step()

                running_loss += loss.item() * inputs.size(0)
                running_corrects += torch.sum(preds == labels.data)
                all_preds.extend(preds.cpu().numpy())
                all_labels.extend(labels.cpu().numpy())

            epoch_loss = running_loss / len(dataloaders[phase].dataset)
            epoch_acc = running_corrects.double() / len(dataloaders[phase].dataset)
            precision = precision_score(all_labels, all_preds, average='macro', zero_division=1)
            recall = recall_score(all_labels, all_preds, average='macro', zero_division=1)

            print(f'{phase.capitalize()} - Loss: {epoch_loss:.4f}, Accuracy: {epoch_acc:.4f}, Precision: {precision:.4f}, Recall: {recall:.4f}')

            if phase == 'val' and epoch_acc > best_acc:
                best_acc = epoch_acc
                best_model_wts = copy.deepcopy(model.state_dict())
                best_cm = confusion_matrix(all_labels, all_preds)

        print(f'Epoch [{epoch+1}/{num_epochs}] completed.\n')

    model.load_state_dict(best_model_wts)
    return model, best_acc, best_cm

# Dataset setup for Tiny ImageNet
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.480, 0.448, 0.398], std=[0.277, 0.269, 0.282])  # Mean and std for Tiny ImageNet
])

# Update the paths to where Tiny ImageNet is stored
data_dir = '/content/tiny-imagenet-200'
train_dir = os.path.join(data_dir, 'train')
val_dir = os.path.join(data_dir, 'val')

# Organize validation set if not already done
val_img_dir = os.path.join(val_dir, 'images')
if os.path.exists(val_img_dir):
    with open(os.path.join(val_dir, 'val_annotations.txt'), 'r') as f:
        for line in f:
            parts = line.strip().split()
            img_name = parts[0]
            class_name = parts[1]

            # Make directory for each class in val
            class_dir = os.path.join(val_dir, class_name)
            if not os.path.exists(class_dir):
                os.makedirs(class_dir)

            # Move each image to the correct class directory
            shutil.move(os.path.join(val_img_dir, img_name), os.path.join(class_dir, img_name))
    os.rmdir(val_img_dir)

# Load train and val datasets
train_dataset = datasets.ImageFolder('/content/tiny-imagenet-200/train', transform=transform)
val_dataset = datasets.ImageFolder('/content/tiny-imagenet-200/val', transform=transform)

# Data loaders - Corrected initialization
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True, num_workers=2)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False, num_workers=2)

dataloaders = {'train': train_loader, 'val': val_loader}

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# Model initialization
model_choice = 'ResNet'
if model_choice == 'ResNet':
    model = models.resnet34(weights='IMAGENET1K_V1').to(device)  # Updated from pretrained=True

    num_features = model.fc.in_features
    model.fc = nn.Linear(num_features, 200).to(device)
    num_epochs = 150
elif model_choice == 'DenseNet':
    model = models.densenet121(weights='IMAGENET1K_V1').to(device)  # Updated from pretrained=True
    # Modify the final layer
    num_features = model.classifier.in_features
    model.classifier = nn.Linear(num_features, 200).to(device)
    num_epochs = 200

# Initialize criterion and optimizer
criterion = CrossEntropyWithSelfDistillation(alpha=0.1, beta=0.05, temperature=3.0)
optimizer = optim.SGD(model.parameters(), lr=0.001, momentum=0.9)

# Train the model
model, best_acc, best_cm = train_model_with_distillation(model, dataloaders, criterion, optimizer, num_epochs=num_epochs, device=device)

print(f'Best Validation Accuracy: {best_acc:.4f}')
print('Confusion Matrix:')
print(best_cm)

# Plot confusion matrix
plt.figure(figsize=(10, 8))
plt.imshow(best_cm, interpolation='nearest', cmap=plt.cm.Blues)
plt.title('Confusion Matrix')
plt.colorbar()
plt.ylabel('True label')
plt.xlabel('Predicted label')
plt.show()


Downloading: "https://download.pytorch.org/models/resnet34-b627a593.pth" to /root/.cache/torch/hub/checkpoints/resnet34-b627a593.pth
100%|██████████| 83.3M/83.3M [00:00<00:00, 107MB/s] 


Epoch [1/150], Alpha: 0.1000, Beta: 0.0500, Temperature: 3.0
Train - Loss: 2.0698, Accuracy: 0.5737, Precision: 0.5784, Recall: 0.5737
Val - Loss: 1.2138, Accuracy: 0.7359, Precision: 0.7491, Recall: 0.7359
Epoch [1/150] completed.

Epoch [2/150], Alpha: 0.2000, Beta: 0.0750, Temperature: 3.0
Train - Loss: 1.1436, Accuracy: 0.7836, Precision: 0.7843, Recall: 0.7836
Val - Loss: 1.2018, Accuracy: 0.7585, Precision: 0.7677, Recall: 0.7585
Epoch [2/150] completed.

Epoch [3/150], Alpha: 0.3000, Beta: 0.1000, Temperature: 3.0
Train - Loss: 0.9680, Accuracy: 0.8592, Precision: 0.8598, Recall: 0.8592
Val - Loss: 1.2245, Accuracy: 0.7633, Precision: 0.7726, Recall: 0.7633
Epoch [3/150] completed.

Epoch [4/150], Alpha: 0.4000, Beta: 0.1250, Temperature: 3.0
Train - Loss: 0.8781, Accuracy: 0.9099, Precision: 0.9104, Recall: 0.9099
Val - Loss: 1.2450, Accuracy: 0.7624, Precision: 0.7691, Recall: 0.7624
Epoch [4/150] completed.

Epoch [5/150], Alpha: 0.5000, Beta: 0.1500, Temperature: 3.0
Train -